# Evaluation Frameworks — RAGAS, LangSmith, DeepEval, Promptfoo, TruLens, Arize Phoenix

Tooling accelerates harnesses, tracing, and dashboards—but you still own rubrics and gates.


## Learning Objectives

- Map frameworks to use cases
- Sketch RAGAS-style metrics inputs
- Wire LangSmith / Promptfoo-style matrices
- Understand TruLens & Phoenix feedback loops


## 1. Framework Map

| Tool | Strength |
|------|----------|
| RAGAS | RAG metric suite |
| LangSmith | Trace + datasets + evals |
| DeepEval | Test-like assertions |
| Promptfoo | Prompt/matrix CI |
| TruLens | Feedback functions |
| Arize Phoenix | Tracing + eval viz |


## 2. RAGAS (Conceptual Usage)

Inputs typically include question, answer, contexts, optionally ground truth.


In [ ]:
# Demo 1 — RAGAS-style metric inputs
samples = [
    {
        "question": "What is the return window?",
        "answer": "30 days with receipt.",
        "contexts": ["Returns accepted within 30 days with receipt."],
        "ground_truth": "30 days",
    }
]
print(samples[0].keys())


## 3. LangSmith

```bash
# export LANGCHAIN_API_KEY=YOUR_LANGSMITH_API_KEY_HERE
# export LANGCHAIN_TRACING_V2=true
```


In [ ]:
# Demo 2 — LangSmith env placeholders
import os
LANGSMITH_API_KEY = os.getenv("LANGCHAIN_API_KEY", "YOUR_LANGSMITH_API_KEY_HERE")
print("tracing key configured:", "YOUR_" not in LANGSMITH_API_KEY)


## 4. DeepEval

Pytest-flavored assertions for relevance, faithfulness, toxicity—good for CI unitization.


## 5. Promptfoo

YAML matrices over prompts × models × tests; great for regression.


In [ ]:
# Demo 3 — Emulate a promptfoo-style matrix in pure Python
import itertools

prompts = ["Be concise: {q}", "Be thorough: {q}"]
models = ["local-7b", "cloud-large"]
questions = ["What is RAG?", "Define TTFT"]
matrix = list(itertools.product(prompts, models, questions))
print("combinations", len(matrix))
print(matrix[0])


## 6. TruLens & Arize Phoenix

Feedback functions + tracing UIs to inspect spans (retrieve, generate, tools).


In [ ]:
# Demo 4 — Toy feedback function interface
from typing import Callable

Feedback = Callable[[str, str], float]

def relevance(question: str, answer: str) -> float:
    q, a = set(question.lower().split()), set(answer.lower().split())
    return len(q & a) / max(len(q), 1)

feedbacks: dict[str, Feedback] = {"relevance": relevance}
print(feedbacks["relevance"]("return window days", "The return window is 30 days"))


### Try it yourself — Frameworks

- Pick one framework and run 10 RAG samples
- Add promptfoo or matrix CI to a toy prompt
- Trace a single request and label failure stage


## Deep Dive Workshop — 06 Evaluation Frameworks

This section expands the notebook into instructor/textbook depth. Work through each subsection: **definition → why it matters → how it works → intuition → pitfalls → when to use**.

```mermaid
flowchart TB
  D[Definition] --> W[Why it matters]
  W --> H[How it works]
  H --> I[Intuition]
  I --> P[Pitfalls]
  P --> U[When to use]
```


### Concept card pack for `06-evaluation-frameworks`

| Concept | Definition | Why it matters | Common pitfall |
|---------|------------|----------------|----------------|
| Primary abstraction | Core object this lesson centers on | Anchors design conversations | Vague naming |
| Quality oracle | How you know the system is right | Prevents demo-driven development | Using vibes only |
| Latency budget | Max user-visible wait | Drives architecture | Ignoring TTFT vs e2e |
| Cost unit | $ per successful task | Makes tradeoffs real | Optimizing tokens not outcomes |
| Trust boundary | Where data/control changes hands | Security design | Treating vendors as internal |
| Feedback loop | How production improves the system | Sustainable quality | No path from thumbs-down to evals |

**Intuition:** If you cannot fill this table for your system, you are not ready to choose models or frameworks.


### Pipeline walkthrough (apply to 06-evaluation-frameworks)

```
1. Input arrives (user / job / webhook)
2. Normalize + authorize + budget check
3. Gather context (files, RAG, tools, memory)
4. Model / deterministic compute
5. Validate output (schema, policy, tests)
6. Side effects (write, ticket, PR) with authz
7. Observe (metrics, traces, feedback)
8. Learn (eval suite growth, prompt/model revision)
```

**When to compress steps:** tiny internal tools. **When to keep all steps:** multi-tenant or regulated production.


### Evaluation advanced notes

Split **unit/deterministic**, **golden**, **judge**, **adversarial**, and **online**.  
Hard-gate safety; soft-gate quality with confidence.  
Private suites beat public leaderboards for ship decisions.

| Stage | Example metric |
|-------|----------------|
| Retrieval | MRR / hit@k |
| Generation | faithfulness |
| Agent | task success + step cost |
| Safety | bypass rate |


In [ ]:
# Extra demo — bootstrap mean for flaky judge scores
import random

def bootstrap_mean(scores, n=500, rng=random.Random(0)):
    means = []
    for _ in range(n):
        sample = [scores[rng.randrange(len(scores))] for _ in scores]
        means.append(sum(sample)/len(sample))
    means.sort()
    return means[int(0.025*n)], sum(means)/len(means), means[int(0.975*n)]

print(bootstrap_mean([0.7,0.8,0.75,0.9,0.6,0.85]))


In [ ]:
# Extra demo — confusion-style agreement human vs judge
def agreement(human, judge):
    assert len(human)==len(judge)
    return sum(h==j for h,j in zip(human,judge))/len(human)

print(agreement([1,1,0,1],[1,0,0,1]))


### Sample interview Q&A — evaluation

**Q:** Offline score is up but users hate the new bot. Why?  
**A:** Distribution shift, metric mismatch, latency regression, or judge bias—slice online feedback by intent and compare traces.

**Q:** How do you stop flaky LLM tests from blocking CI?  
**A:** Deterministic gates on PR; statistical gates nightly; pin seeds/temperature; quarantine; never ignore security suites.


### Comparison matrix exercise

Fill this for two competing designs in this topic:

| Dimension | Option A | Option B | Winner / why |
|-----------|----------|----------|--------------|
| Latency | | | |
| Cost at 10× scale | | | |
| Quality risk | | | |
| Ops burden | | | |
| Security / privacy | | | |
| Time to MVP | | | |


In [ ]:
# Workshop demo — decision scorecard
from dataclasses import dataclass

@dataclass
class Option:
    name: str
    latency: int  # 1=best .. 5=worst
    cost: int
    quality_risk: int
    ops: int
    security: int

def score(o: Option, weights=None) -> float:
    weights = weights or dict(latency=1, cost=1, quality_risk=2, ops=1, security=2)
    return (
        o.latency*weights['latency'] + o.cost*weights['cost'] +
        o.quality_risk*weights['quality_risk'] + o.ops*weights['ops'] +
        o.security*weights['security']
    )

a = Option('A', 2, 3, 2, 2, 2)
b = Option('B', 3, 1, 3, 4, 2)
print(a.name, score(a), b.name, score(b), '-> prefer', a.name if score(a)<score(b) else b.name)


In [ ]:
# Workshop demo — experiment log (use while studying this notebook)
from dataclasses import dataclass, asdict
import json, time

@dataclass
class Experiment:
    hypothesis: str
    setup: str
    metric: str
    baseline: float | None = None
    treatment: float | None = None
    notes: str = ''
    ts: float = 0.0

    def __post_init__(self):
        if not self.ts:
            self.ts = time.time()

exp = Experiment(
    hypothesis='Technique from this lesson improves the primary metric',
    setup='Describe fixtures / model / dataset version',
    metric='name of metric',
    baseline=0.0,
    treatment=0.0,
)
print(json.dumps(asdict(exp), indent=2))


### ASCII architecture sketch template

```
[ Clients ]
     |
[ Edge / API Gateway ] -- authn/z, rate limit
     |
[ Orchestration ] ------+-- prompts / policies
     |                  +-- eval hooks
     +-- context layer (RAG / tools / memory)
     |
[ Model interface ] ---- local and/or cloud
     |
[ Data plane ] --------- indexes, OLTP, object store
     |
[ Observability ] ------ logs, metrics, traces, feedback
```

Copy into your notes and annotate trust boundaries with `***`.


### Pitfalls clinic (read aloud)

1. **Metric theater** — optimizing a proxy that users don't feel  
2. **Context stuffing** — more tokens ≠ more truth  
3. **Prompt as security** — never the only control  
4. **Hidden coupling** — tools/models/indexes version-drift  
5. **No rollback** — can't revert prompt/model quickly  
6. **Eval contamination** — testing on training-like snippets  
7. **Happy-path demos** — skipping adversarial & empty-retrieve cases  


### Try it yourself — extended set

1. Teach the top 3 ideas from this notebook to a rubber duck in 5 minutes  
2. Write 5 quiz questions (with answers) for a junior engineer  
3. Implement one code demo with a real dependency (API or local model) using env placeholders  
4. Break a naive design on purpose; list the failure mode and the fix  
5. Add two rows to your personal glossary with examples from work  
6. Produce a one-page cheat sheet you could use in an interview  


### Mini case study

**Scenario:** Leadership wants this capability in production in six weeks with two engineers.

**Your job:** Propose an MVP that keeps irreversible risks controlled, names the eval gates, and lists what you explicitly defer.

Deliverable structure:
- MVP user story  
- Non-goals  
- Architecture (6 boxes max)  
- Eval gate table  
- Risk register (top 5)  
- Week-by-week plan  


In [ ]:
# Case study helper — risk register
import pandas as pd

risks = pd.DataFrame([
    {'risk': 'quality_miss', 'likelihood': 3, 'impact': 3, 'mitigation': 'golden evals + canary'},
    {'risk': 'cost_overrun', 'likelihood': 3, 'impact': 2, 'mitigation': 'budgets + cache'},
    {'risk': 'data_leak', 'likelihood': 2, 'impact': 5, 'mitigation': 'ACL + redaction'},
    {'risk': 'prompt_injection', 'likelihood': 4, 'impact': 4, 'mitigation': 'boundaries + allowlists'},
    {'risk': 'ops_pages', 'likelihood': 3, 'impact': 3, 'mitigation': 'runbooks + rollback'},
])
risks['score'] = risks.likelihood * risks.impact
print(risks.sort_values('score', ascending=False).to_string(index=False))


### Interview drill (topic-local)

Use the STAR or design template. Timebox 8 minutes.

**Prompt:** “Walk me through how you would productionize the main idea of this notebook.”

Checklist for a strong answer:
- [ ] Clarifying questions  
- [ ] Constraints & numbers  
- [ ] Diagram  
- [ ] Deep dive on hardest part  
- [ ] Evals  
- [ ] Security  
- [ ] Rollout / rollback  


### Glossary boost

| Term | Expanded meaning |
|------|------------------|
| Canary | Partial traffic to a new variant with automatic rollback |
| Golden set | Versioned labeled examples for regression |
| TTFT | Time to first token — interactive UX driver |
| Packing | Selecting/ordering context under a token budget |
| HITL | Human approval inserted before side effects |
| Idempotency | Safe retries without duplicate side effects |
| Shadow traffic | New system sees traffic but doesn't affect users |
| Circuit breaker | Stop calling a failing dependency temporarily |


In [ ]:
# Self-check quiz (run and answer mentally before printing answers)
QUESTIONS = [
    'What oracle proves success for this topic?',
    'Name one metric that can be gamed and a better alternative.',
    'What is the top security failure mode?',
    'What would you defer in an MVP?',
    'How do you rollback a bad change here?',
]
for i, q in enumerate(QUESTIONS, 1):
    print(f'Q{i}. {q}')
print('\n--- suggested answer hints ---')
HINTS = [
    'executable tests / task success / human rubric',
    'longer answers != better; use task success',
    'trust boundary crossing / injection / ACL',
    'multi-agent, perfect UI, every connector',
    'versioned prompts/models + traffic switch',
]
for h in HINTS:
    print('-', h)


### Further practice roadmap for `06-evaluation-frameworks`

| Horizon | Action |
|---------|--------|
| Today | Re-run all code cells; note questions |
| This week | Apply one technique to a real repo/service |
| This month | Add an eval or security test covering this topic |
| Interview ready | Give a 10-minute teach-back with a diagram |


## Lab: End-to-end scenario

Work this scenario in your notes, then implement the smallest possible spike.

### Scenario brief
A team wants to adopt the techniques from this notebook for a **real internal tool** used daily by 200 people. Leadership cares about reliability and auditability more than flashy demos.

### Deliverables
1. One-paragraph problem statement  
2. Success metrics (3) with oracles  
3. Architecture sketch with trust boundaries  
4. Threats / failure modes (5)  
5. Eval plan (offline + online)  
6. 2-week MVP scope and explicit non-goals  

### Review questions
- What happens when context is empty?  
- What happens when the model is down?  
- What happens when a user is malicious?  
- How do you prove a release is safer/better than last week?  


In [ ]:
# Lab helper — MVP scope tracker
from dataclasses import dataclass, field

@dataclass
class MVP:
    must: list[str] = field(default_factory=list)
    should: list[str] = field(default_factory=list)
    defer: list[str] = field(default_factory=list)

    def show(self):
        for label, items in [('MUST', self.must), ('SHOULD', self.should), ('DEFER', self.defer)]:
            print(label)
            for i in items:
                print(' -', i)

mvp = MVP(
    must=['core happy path', 'authn', 'basic eval smoke', 'rollback switch'],
    should=['streaming UX', 'dashboards'],
    defer=['multi-agent', 'perfect personalization', 'every connector'],
)
mvp.show()


## Operator runbook sketch

| Symptom | Likely cause | First checks | Mitigation |
|---------|--------------|--------------|------------|
| Latency spike | Downstream model / retrieve | p95 by stage, saturation | shed load, failover |
| Quality drop | Prompt/model/index change | diff versions, eval slice | rollback |
| Cost spike | loops / huge prompts | tokens/req, step counts | budget breaker |
| Security alert | injection / ACL | traces + retrieved IDs | kill switch |

Keep this table in your ops wiki; customize per system.


In [ ]:
# Operator helper — stage latency rollup
from statistics import mean

stages = {
    'gateway': [20, 25, 22],
    'retrieve': [80, 120, 95],
    'generate': [900, 1100, 980],
}
for k, v in stages.items():
    print(f'{k:10} mean={mean(v):.0f}ms max={max(v)}ms')
print('e2e~', sum(mean(v) for v in stages.values()), 'ms')


## Teaching notes (for study groups)

- Start with the comparison table; argue both sides for 5 minutes  
- Pair-program one demo cell with a real endpoint (placeholder keys)  
- Each person writes one failure case the suite must catch  
- End with a 60-second summary of when *not* to use the technique  


## Summary & Key Takeaways

- Frameworks help; rubrics and datasets still decide quality
- Use matrices for prompt/model regressions
- Tracing connects metrics to root causes
